# micrograd exercises



In [ ]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [1]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29
from math import sin,cos , sqrt
def gradf(a, b, c):
   return [
        -3*a**2 - 1/(2*sqrt(a)),   # df/da
        3*cos(3*b) + 2.5*b**1.5,   # df/db
        1/c**2                     # df/dc
    ]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [10]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell

# -----------

from math import sin, cos

def f(a, b, c):
    return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5
h=0.000001
numerical_grad = [(f(2+h, 3, 4)-f(2,3,4))/h,
                  (f(2, 3+h, 4)-f(2,3,4))/h,
                  (f(2, 3, 4+h)-f(2,3,4))/h
                  ]


     # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353559348809995
OK for dim 1: expected 10.25699027111255, yours returns 10.256991666679482
OK for dim 2: expected 0.0625, yours returns 0.062499984743169534


In [24]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.
from math import sin, cos

def f(a, b, c):
    return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5
h=0.000001
numerical_grad2 = [(f(2+h, 3, 4)-(f(2-h, 3, 4)))/(2*h),
                  (f(2, 3+h, 4)-(f(2,3-h,4)))/(2*h),
                  ((f(2, 3, 4+h))-(f(2, 3, 4-h)))/(2*h)
                  ]
for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553391353245
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027401572
OK for dim 2: expected 0.0625, yours returns 0.06250000028629188


## section 2: support for softmax

In [26]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out


  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out=Value(self.data*other.data,(self,other),'*')
    def _backward():
      self.grad += other.data*out.grad
      other.grad += self.data*out.grad
    out._backward=_backward
    return out

  def __truediv__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out=Value(self.data/other.data,(self,other),'/')
    def _backward():
      self.grad += 1/other.data*out.grad
      other.grad += -self.data/other.data**2*out.grad
    out._backward=_backward
    return out


  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
      self.grad += other * self.data**(other-1) * out.grad
    out._backward = _backward

    return out

  def exp(self):
    x = self.data
    out = Value(exp(x), (self, ), 'exp')

    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward

    return out

  def log(self):
    x = self.data
    out = Value(log(x), (self, ), 'log')

    def _backward():
      self.grad += 1/x * out.grad
    out._backward = _backward

    return out

  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here
  # TODO
  # ------

  def backward(self): # exactly as in video
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

chatgpt changes


In [27]:
from math import exp, log

class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    # -------------------------
    # ADDED
    # Needed for: sum(counts)
    # because sum starts with 0
    # -------------------------
    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    # -------------------------
    # ADDED
    # Makes number * Value work
    # -------------------------
    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data / other.data, (self, other), '/')

        def _backward():
            self.grad += (1 / other.data) * out.grad
            other.grad += (-self.data / (other.data ** 2)) * out.grad

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers"

        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    # -------------------------
    # ADDED
    # Needed for:
    # loss = -probs[3].log()
    # -------------------------
    def __neg__(self):
        return self * -1

    # Optional convenience methods
    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def exp(self):
        x = self.data
        out = Value(exp(x), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward
        return out

    def log(self):
        x = self.data
        out = Value(log(x), (self,), 'log')

        def _backward():
            self.grad += (1 / x) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

Extend the Value class by implementing the missing mathematical operations, each with both its forward computation and backward (gradient) rule, so the rest of the notebook can build and differentiate computation graphs automatically.

In [28]:
# without referencing our code/video __too__ much, make this cell work

# you'll have to implement (in some cases re-implemented) a number of functions

# of the Value object, similar to what we've seen in the video.

# instead of the squared error loss this implements the negative log likelihood

# loss, which is very often used in classification.



# this is the softmax function

# https://en.wikipedia.org/wiki/Softmax_function

def softmax(logits):

  counts = [logit.exp() for logit in logits]

  denominator = sum(counts)

  out = [c / denominator for c in counts]

  return out



# this is the negative log likelihood loss function, pervasive in classification

logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]

probs = softmax(logits)

loss = -probs[3].log() # dim 3 acts as the label for this input example

loss.backward()

print(loss.data)



ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]

for dim in range(4):

  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'

  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


2.1755153626167147
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.886450380640099


# Micrograd Exercise - Building the `Value` Class (Softmax & NLL)

## Goal

The goal of this exercise was **NOT** to learn Softmax or Negative Log Likelihood.

The real goal was to **extend our `Value` class** so it can support more mathematical operations and automatically compute gradients for them.

---

# What did we build?

Previously our `Value` class only knew how to do:

- Addition (`+`)

Now we taught it to perform:

- Multiplication (`*`)
- Division (`/`)
- Power (`**`)
- Exponential (`exp`)
- Logarithm (`log`)
- Unary Negation (`-value`)
- Reverse Addition (`0 + value`)

Each operation has two jobs:

1. Compute the forward value.
2. Know how to send gradients backward.

Example:

For multiplication,

Forward:

```
a * b
```

Backward:

```
da = b
db = a
```

This pattern is repeated for every new operation.

---

# Understanding the Softmax Code

We were given:

```python
counts = [logit.exp() for logit in logits]
```

This converts every logit into an exponential value.

Example:

```
2
↓
exp(2)
```

---

Then:

```python
denominator = sum(counts)
```

This adds all exponential values together.

We needed `__radd__` because Python internally does:

```python
0 + counts[0]
```

Without `__radd__`, this fails.

---

Next,

```python
out = [c / denominator for c in counts]
```

Each exponential value is divided by the total sum.

This converts them into probabilities.

This required:

- `__truediv__`

---

Then,

```python
loss = -probs[3].log()
```

This computes the Negative Log Likelihood Loss.

It required:

- `log()`
- `__neg__`

---

Finally,

```python
loss.backward()
```

This starts backpropagation.

Every operation (`+`, `*`, `/`, `exp`, `log`, etc.) already knows how to pass gradients to its parents.

Because of that, gradients automatically flow all the way back to every input logit.

---

# Special Python Cases

## Why did we need `__radd__`?

Python's `sum()` starts like this:

```python
0 + first_value
```

`0` is an integer.

Integers don't know how to add a `Value`.

Python then asks the `Value` object:

> "Can YOU handle this?"

That is exactly what `__radd__()` does.

---

## Why did we need `__neg__`?

When Python sees:

```python
-value
```

it looks for:

```python
__neg__()
```

Without it,

```python
-value
```

throws an error.

---

# Biggest Lesson

Every mathematical operation inside our autodiff engine has two responsibilities.

## Forward Pass

Compute the value.

Example:

```
2 * 3 = 6
```

---

## Backward Pass

Compute how gradients should flow.

Example:

```
d(a*b)/da = b
d(a*b)/db = a
```

This is exactly how PyTorch works internally.

---

# Mental Model

Think of every operation as a small machine.

```
Inputs
   │
   ▼
Operation
   │
   ▼
Output
```

The machine knows:

- How to produce the output.
- How to send gradients back to its inputs.

Connecting many of these machines together forms the computational graph.

Calling

```python
loss.backward()
```

simply tells every machine:

> "Pass the gradients to your parents."

---

# What I Learned

- A `Value` object stores:
  - data
  - gradient
  - parents
  - backward function

- Every new mathematical operation needs:
  - Forward computation
  - Backward gradient rule

- Python operators (`+`, `*`, `/`, `-`) are implemented using magic methods like:
  - `__add__`
  - `__mul__`
  - `__truediv__`
  - `__neg__`
  - `__radd__`

- Softmax itself was not the focus.

- The exercise was really about making our `Value` class powerful enough to support more operations while preserving automatic differentiation.

---

# Key Takeaway

We are no longer just **using** automatic differentiation.

We are **building** an automatic differentiation engine, one mathematical operation at a time.

In [29]:
import torch

# Create PyTorch tensors.
# requires_grad=True tells PyTorch:
# "Track every operation so gradients can be computed later."
logits = torch.tensor([0.0, 3.0, -2.0, 1.0], requires_grad=True)

# Step 1: Compute exponentials (same as logit.exp() in our Value class)
counts = logits.exp()

# Step 2: Sum all exponentials
denominator = counts.sum()

# Step 3: Compute softmax probabilities
probs = counts / denominator

# Step 4: Compute Negative Log Likelihood Loss
# We assume class index 3 is the correct label.
loss = -torch.log(probs[3])

# Step 5: Backpropagation
# PyTorch automatically computes gradients.
loss.backward()

# Print loss value
print(loss.item())

# Print gradients for each logit
print(logits.grad)

# Expected gradients (same as micrograd)
ans = [
    0.041772570515350445,
    0.8390245074625319,
    0.005653302662216329,
    -0.8864503806400986
]

# Compare PyTorch gradients with expected gradients
for dim in range(4):
    ok = "OK" if abs(logits.grad[dim].item() - ans[dim]) < 1e-5 else "WRONG!"
    print(
        f"{ok} for dim {dim}: "
        f"expected {ans[dim]}, "
        f"torch returned {logits.grad[dim].item()}"
    )

2.1755154132843018
tensor([ 0.0418,  0.8390,  0.0057, -0.8865])
OK for dim 0: expected 0.041772570515350445, torch returned 0.041772566735744476
OK for dim 1: expected 0.8390245074625319, torch returned 0.8390244245529175
OK for dim 2: expected 0.005653302662216329, torch returned 0.005653302185237408
OK for dim 3: expected -0.8864503806400986, torch returned -0.8864503502845764
